### Install new libraries

In [15]:
#!pip install ddgs trafilatura

### Setup

In [16]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from pprint import pprint
from IPython.display import display, Markdown
from ddgs import DDGS
import trafilatura


# ------------------------------
# Setup
# ------------------------------

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
	raise Exception("API key is missing")
client = OpenAI()

MODEL = "gpt-4.1-mini"


### Step 1: Define the Tools

In [17]:
def search_web(query: str):
	"""Search the web using DuckDuckGo browser. Returns 3 results."""
	ddgs = DDGS()
	results = ddgs.text(query, max_results=3)
	print(f"  \u2705 Got results")
	return json.dumps(results, indent=2)

In [18]:
def fetch_url(url: str):
	"""Fetch the content of a URL using trafilatura."""
	downloaded = trafilatura.fetch_url(url)
	if downloaded:
		text =  trafilatura.extract(downloaded)
		if text:
			print(f"  \u2705 Got text: {len(text)} characters")
			return text
	print(f"  \u274C Failed to fetch or extract text.") # For debugging
	return f"Could not extract text from {url}. Try a different source." # For LLM input feedback


In [19]:
#Test search_web function with a sample query
search_web("Apples in Madagascar") #Calling the search_web function to test it with a sample query(str)

#search_web("AI in healthcare in 2030") #Calling the search_web function to test it with a sample query(str)

  ✅ Got results


'[\n  {\n    "title": "Field Guide to Fruit on Madagascar - jurgenenkatja",\n    "href": "https://www.jurgenenkatja.nl/en/fruit-on-madagascar/",\n    "body": "English: Mimusops coriacea, Monkey\'s Apple, Mulch Apple Malagasy: Anganahara, Voranto, Vonanto Dutch: Mimusops coriacea It is also called the Malagasy apple. The outside is hard and you have to work to break the fruit open. This species originates from the Comoros and Madagascar. We came across this fruit on the east coast, but it also seems to grow, to a lesser extent, in the northwest. The ..."\n  },\n  {\n    "title": "The list of fruits to eat in Madagascar - explurt.com",\n    "href": "https://explurt.com/blog-english/madagascar-english/the-list-of-fruits-to-eat-in-madagascar/",\n    "body": "The fruit of love, the apple is also found in Madagascar, in many different varieties. You can enjoy both green and red apples during your stay, and are sure to eat them for breakfast and dessert."\n  },\n  {\n    "title": "Apple Culti

In [20]:
#Test fetch_url function with a sample URL
result = fetch_url("https://explurt.com/blog-english/madagascar-english/the-list-of-fruits-to-eat-in-madagascar/") #Calling the fetch_url function to test it with a sample URL(str)
print(result)

  ✅ Got text: 5756 characters
The list of fruits to eat in Madagascar
A country of 586,884 km2, the Red Island, the Big Island or the Island of the Indian Ocean, Madagascar is a land of many facets and flavours. From its disjointed sides, the island hides fruits with singular smells and tastes. From oranges to jujubes, the island harbours the entire fruity repertoire of tropical countries; between the Tropic of Capricorn and the Equator, Madagascar is a place where all kinds of fruit germinate and can be savoured.
In this article, we take a look at the list of fruits found in Madagascar. Like a travel guide, we draw up our list of fruits to be encountered each month in Madagascar.
January :
Apricot
First cultivated in China, the apricot is a sweet orange fruit shaped like a ball with a slit in the middle. The fruit is found in January in Madagascar. The flesh is sweet and has a high provitamin A content. The fruit ripens after picking.
February :
Plums
Ideal for stomach aches and gastr

### Step 2: Describe as LLM tools

In [21]:
tools = []

In [22]:
#Describe search_web function as an LLM tool
search_web_function = {
	"name": "search_web",
	"description": "Search the web using DuckDuckGo browser. Returns 3 results.",
	"parameters": {
		"type": "object",
		"properties": {
			"query": {
				"type": "string",
				"description": "The search query to find relevant websites."
			}
		},
		"required": ["query"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": search_web_function})

In [23]:
#Describe fetch_url function as an LLM tool
fetch_url_function = {
	"name": "fetch_url",
	"description": "Fetch and extract the main content from a web page.",
	"parameters": {
		"type": "object",
		"properties": {
			"url": {
				"type": "string",
				"description": "The URL of the web page to fetch and extract text from."
			}
		},
		"required": ["url"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": fetch_url_function})


In [24]:
#Check the tools list
tools

[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'Search the web using DuckDuckGo browser. Returns 3 results.',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query to find relevant websites.'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'Fetch and extract the main content from a web page.',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'The URL of the web page to fetch and extract text from.'}},
    'required': ['url']}}}]

### Step 3: Tool Call handler

In [25]:
def handle_tool_call(tool_calls):
	tool_results = []


	for tool_call in tool_calls:  # handle every tool call the model requested
		function_name = tool_call.function.name
		args = json.loads(tool_call.function.arguments)  # arguments are JSON
		print(f"  \u0001F527 Calling function {function_name} with arguments {args}") # For future debugging

		#Route to the appropriate function based on function_name
		if function_name == "search_web":
			result = search_web(args["query"])
			content = f"Search results: {result}"

		elif function_name == "fetch_url":
			result = fetch_url(args["url"])
			content = f"Fetched URL content: {result}"
		else:
			content = f"Unknown function: {function_name}"
		tool_call_result = {
			"role": "tool",
			"content": content,
			"tool_call_id": tool_call.id
		}
		tool_results.append(tool_call_result)
	return tool_results

### Step 4: The System Prompt

 This tells the LLM who it is and how to behave. The key things:
- What its job is
- What tools it has
- What process to follow
- What output format to produce




In [26]:

# RESEARCH_AGENT_PROMPT = """You are a research agent that can search the web and fetch content from URLs.
# You have access to the following tools:
# 1. search_web(query: str) -> str: Search the web using DuckDuckGo browser. Returns 3 results.
# 2. fetch_url(url: str) -> str: Fetch and extract the main content from a web page.
# Your task is to answer questions and provide information by using these tools when necessary.\
# If you need to search for information, use the search_web tool. If you find a relevant URL, use the fetch_url tool to extract the content.\
# Always provide accurate and concise information in your responses. If you cannot find the information, respond with 'I could not find the information you requested.\
# Do not make up information. Always cite your sources when providing information from the web.\
# If you use the fetch_url tool, provide a summary of the content you extracted."""

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""


### Step 5: The Agentic Loop

In [27]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
    """
    Run the research agent on a topic and return the research brief.
    Args:
        topic: The topic to research
        max_iterations: Safety limit to prevent infinite loops
    Returns:
        The research brief as a string
    """
    print(f"\n🔍 Starting research on: {topic}")
    print("=" * 60)

    # Initialize conversation messages list with system prompt + research task
    messages = [
        {"role": "system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and produce a comprehensive research brief:\n {topic}"}
    ]

    # LOOP
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        print(f"\n---Iteration {iteration} ---")

        # 1. Call the LLM and get response
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

        message = response.choices[0].message
        messages.append(message)

        # 2. Check if LLM called tools
        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)

        # 3. Otherwise: no tools were called, read message content
        else:
            content = message.content # No tool calls, so there's just a normal text response from the LLM
            # Check if DONE:, then return
            if content and content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()  # Remove "DONE:" prefix and whitespace
                print(f"  ✅ Research complete!")
                return research_brief

            # Otherwise: not yet done
            else:
                print(f"  💭 Agent is thinking...")
                pprint(content)
                # Loop continues to next iteration

        # 4. If we're entering the final iteration, force a final answer
        if iteration == max_iterations - 1:
            print("⚠️ Safety limit reached. Stopping research in next iteration.")
            messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please deliver your final research brief now, even if it is incomplete."})

    # Fallback return
    return "Research incomplete. Maximum iterations reached without finalizing brief."

### Let's Run IT!

In [28]:
brief = run_research_agent("The impact of AI on healthcare in 2030")
display(Markdown(f"### Research Brief on 'The impact of AI on healthcare in 2030'\n\n{brief}"))


🔍 Starting research on: The impact of AI on healthcare in 2030

---Iteration 1 ---
  F527 Calling function search_web with arguments {'query': 'impact of AI on healthcare in 2030'}
  ✅ Got results

---Iteration 2 ---
  F527 Calling function fetch_url with arguments {'url': 'https://www.kindo.ai/blog/how-ai-will-change-healthcare-by-2030'}
  ✅ Got text: 13481 characters
  F527 Calling function fetch_url with arguments {'url': 'https://www.eicta.iitk.ac.in/knowledge-hub/artificial-intelligence/future-of-ai-in-healthcare-predictions-innovations-2030'}
  ✅ Got text: 9915 characters
  F527 Calling function fetch_url with arguments {'url': 'https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/'}
  ❌ Failed to fetch or extract text.

---Iteration 3 ---
  F527 Calling function search_web with arguments {'query': 'AI healthcare 2030 HIMSS pulse report summary'}
  ✅ Got results

---Iteration 4 ---
  F527 Calling 

### Research Brief on 'The impact of AI on healthcare in 2030'

Research Brief: The Impact of AI on Healthcare in 2030

Key Facts and Statistics:
- By 2030, AI will be foundational in healthcare, integrating deeply into diagnostics, patient management, drug discovery, and clinical decision-making.
- There will be a significant shift from reactive treatment to proactive and preventive care via AI technologies.
- AI-powered medical imaging will provide highly accurate, earlier diagnoses, reducing the need for invasive procedures.
- Predictive AI models will analyze diverse data (EHRs, genomics, lifestyle, environment) to forecast disease onset and personalize care, improving outcomes and reducing readmissions.
- AI-driven virtual assistants and telehealth platforms will democratize access, enabling care for remote and underserved populations.
- The population aged 60 and above is expected to double by 2050, amplifying the need for AI-supported healthcare solutions like virtual nursing and intelligent monitoring.
- AI technologies such as large language models and generative AI will augment administrative workflows and clinical decision support.
- Challenges include data privacy, algorithmic bias, ethical governance, and ensuring trust and transparency.

Main Themes and Arguments:
1. Predictive and Preventive Care: AI’s ability to forecast health risks will shift healthcare from reactive treatment to proactive prevention, tailoring interventions based on individual risk profiles.
2. Precision Diagnostics: AI algorithms will improve diagnostic speed and accuracy in medical imaging and biomarker analyses, supporting personalized treatment planning.
3. Enhanced Healthcare Workforce: AI assistants and robotics will reduce clinician workloads, improve procedural precision, and enable continuous patient monitoring.
4. Increased Accessibility and Equity: AI-driven telemedicine, language translation, and educational tools will bridge gaps in healthcare access globally.
5. Ethical and Regulatory Considerations: Emphasis on privacy safeguards, eliminating bias, maintaining human oversight, and ethical AI deployment is critical for broad adoption.

Notable Data Points:
- AI-assisted diagnosis in areas like oncology and neurology is expected to become routine.
- Semi-autonomous surgical robots will assist with high-precision tasks to reduce complications and speed recovery.
- Technologies such as 5G/6G and edge computing will enable real-time monitoring and data transmission.
- AI applications will move beyond individual care to support public health initiatives through predictive modeling and resource allocation.

Sources:
- https://www.kindo.ai/blog/how-ai-will-change-healthcare-by-2030
- https://www.eicta.iitk.ac.in/knowledge-hub/artificial-intelligence/future-of-ai-in-healthcare-predictions-innovations-2030
- https://www.briefs.co/news/ieee-report-ai-will-pivot-healthcare-to-preventive-care-by-2/
- https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/

Conclusion:
AI is on a path to revolutionize healthcare by 2030 through enhanced disease prediction, precision diagnostics, patient-centric personalized medicine, and improved operational efficiencies. The technology promises not only to improve individual health outcomes but also to reshape healthcare delivery globally by making care more accessible and proactive. Ethical implementation and governance will be key to unlocking AI’s full potential in this sector.